# Analisis de embeddings latentes Aβ/SPECT

Este notebook analiza los embeddings generados durante la fase 1 del entrenamiento, guardados en `data/latent_Abeta`.

Objetivos:
- cargar los latentes `.nii` / `.nii.gz`;
- asociarlos con la clase en `data_info/data_info.csv`;
- revisar balance de clases y disponibilidad de archivos;
- proyectar los embeddings con PCA, t-SNE y UMAP;
- visualizar cada metodo en 2D y 3D;
- calcular metricas exploratorias simples de separacion entre clases.

Suposicion de etiquetas: `0 = control`, `1 = parkinson`. Si el mapeo real es distinto, cambia `LABEL_NAMES` en la celda de configuracion.

## 0. Dependencias opcionales

Ejecuta esta celda si el kernel no tiene las librerias de visualizacion/reduccion dimensional. En el contenedor base detectamos que pueden faltar `scikit-learn`, `umap-learn`, `plotly` y `matplotlib`.

In [ ]:
# Descomenta si hacen falta dependencias en el kernel de Jupyter.
%pip install scikit-learn umap-learn matplotlib plotly seaborn

In [1]:
import numpy as np
import os
import matplotlib.pyplot as plt

In [ ]:
from pathlib import Path

def find_repo_root() -> Path:
    p = Path.cwd().resolve()
    for parent in [p, *p.parents]:
        if (parent / "config.py").is_file():
            return parent
    raise FileNotFoundError("Could not find repo root (config.py)")

PROJECT_ROOT = find_repo_root()
INFER_RUN = PROJECT_ROOT / "result/exp_1/inference/20260508_160032/representations"
file = INFER_RUN / "train/3102.npz"
data = np.load(file)
synthetic_spect_flat = data["sampled_latent_final"]

print(type(synthetic_spect_flat))
print(synthetic_spect_flat.shape)
print(synthetic_spect_flat.dtype)

In [ ]:
volume = synthetic_spect_flat[0]
slice_idx = 20

plt.figure(figsize=(5, 5))
plt.imshow(volume[slice_idx, :, :], cmap="gray")
plt.colorbar()
plt.title(f"sampled_latent_final - slice {slice_idx}")
plt.axis("off")
plt.show()

In [ ]:
len(os.listdir(INFER_RUN / "test"))

## 1. Configuracion e imports

In [ ]:
from pathlib import Path
import warnings

import nibabel as nib
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
try:
    import nbformat  # required by Plotly inside notebooks
    import plotly.express as px
    HAS_PLOTLY = True
except ImportError:
    px = None
    HAS_PLOTLY = False

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score

try:
    import umap
except ImportError as exc:
    raise ImportError('Falta umap-learn. Ejecuta: %pip install umap-learn') from exc

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')

def find_repo_root() -> Path:
    p = Path.cwd().resolve()
    for parent in [p, *p.parents]:
        if (parent / "config.py").is_file():
            return parent
    raise FileNotFoundError("Could not find repo root (config.py)")

PROJECT_ROOT = find_repo_root()
LATENT_DIR = PROJECT_ROOT / 'data' / 'latent_Abeta'
DATA_INFO = PROJECT_ROOT / 'data_info' / 'data_info.csv'

LABEL_NAMES = {
    0: 'control',
    1: 'parkinson',
}

RANDOM_STATE = 42
PCA_PREPROCESS_COMPONENTS = 50
TSNE_PERPLEXITY = 30
UMAP_N_NEIGHBORS = 15
UMAP_MIN_DIST = 0.1

print('Latent dir:', LATENT_DIR)
print('Data info:', DATA_INFO)

## 2. Cargar metadata y latentes

Cada archivo en `data/latent_Abeta` se identifica por el nombre del archivo, por ejemplo `3102.nii.gz -> ID=3102`. Solo se usan latentes que tengan etiqueta en `data_info.csv`.

In [ ]:
def latent_id(path: Path) -> str:
    name = path.name
    if name.endswith('.nii.gz'):
        return name[:-7]
    return path.stem


def load_latent(path: Path) -> np.ndarray:
    data = nib.load(str(path)).get_fdata(dtype=np.float32)
    return np.asarray(data, dtype=np.float32)


info = pd.read_csv(DATA_INFO, dtype={'ID': str})
info['label'] = info['label'].astype(int)
info['class_name'] = info['label'].map(LABEL_NAMES).fillna(info['label'].astype(str))

latent_paths = sorted(list(LATENT_DIR.glob('*.nii')) + list(LATENT_DIR.glob('*.nii.gz')))
latent_df = pd.DataFrame({
    'ID': [latent_id(path) for path in latent_paths],
    'latent_path': latent_paths,
})

merged = latent_df.merge(info, on='ID', how='left', indicator=True)
missing_labels = merged[merged['_merge'] != 'both']
if len(missing_labels):
    print('Latentes sin etiqueta en data_info.csv:')
    display(missing_labels[['ID', 'latent_path']])

merged = merged[merged['_merge'] == 'both'].drop(columns=['_merge']).reset_index(drop=True)
display(merged.head())
print(f'Latentes encontrados: {len(latent_paths)}')
print(f'Latentes con etiqueta: {len(merged)}')

In [ ]:
merged.groupby('label').size()

## 3. Control de calidad inicial

In [ ]:
print('Distribucion por clase:')
display(merged.groupby(['label', 'class_name']).size().rename('n').reset_index())

if 'split' in merged.columns:
    print('Distribucion por split y clase:')
    display(merged.groupby(['split', 'class_name']).size().rename('n').reset_index())

sample_shapes = []
for path in merged['latent_path'].head(min(10, len(merged))):
    sample_shapes.append(load_latent(path).shape)
print('Shapes observados en muestra:', sorted(set(sample_shapes)))

## 4. Construir matriz de embeddings

Se aplana cada volumen latente 3D a un vector. Para t-SNE y UMAP se usa una reduccion PCA previa a maximo 50 componentes, una practica comun para reducir ruido y costo computacional.

In [ ]:
latents = []
shapes = []
for path in merged['latent_path']:
    volume = load_latent(path)
    shapes.append(volume.shape)
    latents.append(volume.reshape(-1))

if len(set(shapes)) != 1:
    raise ValueError(f'Los latentes no tienen todos el mismo shape: {sorted(set(shapes))}')

X = np.stack(latents).astype(np.float32)
y = merged['label'].to_numpy()
class_names = merged['class_name'].to_numpy()

print('Shape latente 3D:', shapes[0])
print('Matriz X:', X.shape)
print('Memoria aproximada X:', f'{X.nbytes / 1024**2:.1f} MB')

scaler = StandardScaler(with_mean=True, with_std=True)
X_scaled = scaler.fit_transform(X)

pre_components = min(PCA_PREPROCESS_COMPONENTS, X_scaled.shape[0] - 1, X_scaled.shape[1])
pca_pre = PCA(n_components=pre_components, random_state=RANDOM_STATE)
X_pre = pca_pre.fit_transform(X_scaled)

print('X_pre para t-SNE/UMAP:', X_pre.shape)
print('Varianza explicada PCA previa:', f'{pca_pre.explained_variance_ratio_.sum():.3f}')

## 5. Funciones de visualizacion y metricas

In [ ]:
def make_projection_df(coords, method):
    cols = [f'{method}_{i+1}' for i in range(coords.shape[1])]
    df = pd.DataFrame(coords, columns=cols)
    df['ID'] = merged['ID'].values
    df['label'] = y
    df['class_name'] = class_names
    if 'split' in merged.columns:
        df['split'] = merged['split'].values
    return df


def plot_2d(df, method):
    x_col, y_col = f'{method}_1', f'{method}_2'
    plt.figure(figsize=(8, 6))
    sns.scatterplot(
        data=df,
        x=x_col,
        y=y_col,
        hue='class_name',
        style='split' if 'split' in df.columns else None,
        s=70,
        alpha=0.85,
    )
    plt.title(f'{method} 2D de embeddings latentes')
    plt.xlabel(x_col)
    plt.ylabel(y_col)
    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.show()


def plot_3d(df, method):
    x_col, y_col, z_col = f'{method}_1', f'{method}_2', f'{method}_3'
    if HAS_PLOTLY:
        fig = px.scatter_3d(
            df,
            x=x_col,
            y=y_col,
            z=z_col,
            color='class_name',
            symbol='split' if 'split' in df.columns else None,
            hover_data=['ID', 'label'] + (['split'] if 'split' in df.columns else []),
            title=f'{method} 3D de embeddings latentes',
            width=850,
            height=650,
        )
        fig.show()
        return

    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d')
    for class_name, group in df.groupby('class_name'):
        ax.scatter(
            group[x_col],
            group[y_col],
            group[z_col],
            label=class_name,
            s=45,
            alpha=0.8,
        )
    ax.set_title(f'{method} 3D de embeddings latentes')
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.set_zlabel(z_col)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()


def projection_metrics(coords, labels, method):
    if len(np.unique(labels)) < 2 or len(labels) < 3:
        return {'method': method, 'silhouette': np.nan, 'davies_bouldin': np.nan}
    return {
        'method': method,
        'silhouette': silhouette_score(coords, labels),
        'davies_bouldin': davies_bouldin_score(coords, labels),
    }

## 6. PCA 2D y 3D

In [ ]:
pca_2d = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X_scaled)
pca_3d = PCA(n_components=3, random_state=RANDOM_STATE).fit_transform(X_scaled)

pca_df_2d = make_projection_df(pca_2d, 'PCA')
pca_df_3d = make_projection_df(pca_3d, 'PCA')

print('Varianza explicada PCA 2D:', PCA(n_components=2, random_state=RANDOM_STATE).fit(X_scaled).explained_variance_ratio_.sum())
print('Varianza explicada PCA 3D:', PCA(n_components=3, random_state=RANDOM_STATE).fit(X_scaled).explained_variance_ratio_.sum())

plot_2d(pca_df_2d, 'PCA')
plot_3d(pca_df_3d, 'PCA')

## 7. t-SNE 2D y 3D

`perplexity` debe ser menor que el numero de muestras. Si hay pocos sujetos, se ajusta automaticamente.

In [ ]:
perplexity = min(TSNE_PERPLEXITY, max(5, (len(X_pre) - 1) // 3))
print('Perplexity usada:', perplexity)

tsne_2d = TSNE(
    n_components=2,
    perplexity=perplexity,
    init='pca',
    learning_rate='auto',
    random_state=RANDOM_STATE,
).fit_transform(X_pre)

tsne_3d = TSNE(
    n_components=3,
    perplexity=perplexity,
    init='pca',
    learning_rate='auto',
    random_state=RANDOM_STATE,
).fit_transform(X_pre)

tsne_df_2d = make_projection_df(tsne_2d, 'TSNE')
tsne_df_3d = make_projection_df(tsne_3d, 'TSNE')

plot_2d(tsne_df_2d, 'TSNE')
plot_3d(tsne_df_3d, 'TSNE')

## 8. UMAP 2D y 3D

In [ ]:
umap_2d = umap.UMAP(
    n_components=2,
    n_neighbors=UMAP_N_NEIGHBORS,
    min_dist=UMAP_MIN_DIST,
    metric='euclidean',
    random_state=RANDOM_STATE,
).fit_transform(X_pre)

umap_3d = umap.UMAP(
    n_components=3,
    n_neighbors=UMAP_N_NEIGHBORS,
    min_dist=UMAP_MIN_DIST,
    metric='euclidean',
    random_state=RANDOM_STATE,
).fit_transform(X_pre)

umap_df_2d = make_projection_df(umap_2d, 'UMAP')
umap_df_3d = make_projection_df(umap_3d, 'UMAP')

plot_2d(umap_df_2d, 'UMAP')
plot_3d(umap_df_3d, 'UMAP')

## 9. Comparacion cuantitativa exploratoria

Estas metricas no sustituyen validacion clinica. Sirven para estimar si las clases forman agrupamientos en cada proyeccion.

- `silhouette`: mayor es mejor, rango aproximado [-1, 1].
- `davies_bouldin`: menor es mejor.

In [ ]:
metrics = pd.DataFrame([
    projection_metrics(pca_2d, y, 'PCA_2D'),
    projection_metrics(pca_3d, y, 'PCA_3D'),
    projection_metrics(tsne_2d, y, 'TSNE_2D'),
    projection_metrics(tsne_3d, y, 'TSNE_3D'),
    projection_metrics(umap_2d, y, 'UMAP_2D'),
    projection_metrics(umap_3d, y, 'UMAP_3D'),
])
display(metrics.sort_values('silhouette', ascending=False))

## 10. Guardar proyecciones para analisis posterior

In [ ]:
OUT_DIR = PROJECT_ROOT / 'result' / 'embedding_analysis'
OUT_DIR.mkdir(parents=True, exist_ok=True)

pca_df_2d.to_csv(OUT_DIR / 'pca_2d.csv', index=False)
pca_df_3d.to_csv(OUT_DIR / 'pca_3d.csv', index=False)
tsne_df_2d.to_csv(OUT_DIR / 'tsne_2d.csv', index=False)
tsne_df_3d.to_csv(OUT_DIR / 'tsne_3d.csv', index=False)
umap_df_2d.to_csv(OUT_DIR / 'umap_2d.csv', index=False)
umap_df_3d.to_csv(OUT_DIR / 'umap_3d.csv', index=False)
metrics.to_csv(OUT_DIR / 'projection_metrics.csv', index=False)

print('Resultados guardados en:', OUT_DIR)

## 11. Clasificacion supervisada de embeddings

En esta seccion se evalua si los embeddings latentes originales contienen informacion discriminativa para separar `control` vs `parkinson`.

Como los embeddings disponibles se tratan aqui como un unico conjunto de entrenamiento, se crea una particion interna estratificada `80/20` para entrenamiento y prueba.

**Importante:** esta clasificacion usa los embeddings en su version original aplanada. No se aplica PCA, t-SNE, UMAP ni ninguna tecnica de reduccion dimensional. Solo se estandarizan las features con `StandardScaler`, ajustado exclusivamente sobre train.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
)

X_train_raw, X_test_raw, y_train, y_test, meta_train, meta_test = train_test_split(
    X,
    y,
    merged,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("Train:", X_train_raw.shape, "Test:", X_test_raw.shape)
print("Features originales por embedding:", X_train_raw.shape[1])
print("Distribucion train:", pd.Series(y_train).map(LABEL_NAMES).value_counts().to_dict())
print("Distribucion test:", pd.Series(y_test).map(LABEL_NAMES).value_counts().to_dict())

### 11.1 Modelos clasicos de ML

Cada modelo usa el mismo preprocesamiento: `StandardScaler -> clasificador`. No hay reduccion dimensional. El scaler se ajusta solo con train.

Para KNN se reporta explicitamente el valor de `K` usado en la columna `n_neighbors`. Se incluye un KNN automatico, que usa el mayor impar hasta 5 permitido por el tamano de train, y un KNN adicional con `K=2` por el conocimiento previo de que existen dos clases.

In [ ]:
def build_pipeline(classifier):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", classifier),
    ])


knn_auto_neighbors = min(5, len(y_train))
if knn_auto_neighbors % 2 == 0:
    knn_auto_neighbors -= 1
knn_auto_neighbors = max(1, knn_auto_neighbors)
knn_k2_neighbors = min(2, len(y_train))

print(f"KNN automatico entrenado con K={knn_auto_neighbors}")
print(f"KNN adicional entrenado con K={knn_k2_neighbors}")

classical_models = {
    "KNN_auto": build_pipeline(KNeighborsClassifier(n_neighbors=knn_auto_neighbors, weights="distance")),
    "KNN_K2": build_pipeline(KNeighborsClassifier(n_neighbors=knn_k2_neighbors, weights="distance")),
    "SVM_RBF": build_pipeline(SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, class_weight="balanced", random_state=RANDOM_STATE)),
    "SVM_linear": build_pipeline(SVC(kernel="linear", C=1.0, probability=True, class_weight="balanced", random_state=RANDOM_STATE)),
    "LogisticRegression": build_pipeline(LogisticRegression(max_iter=5000, class_weight="balanced", random_state=RANDOM_STATE)),
}


def positive_scores(model, X_eval):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X_eval)[:, 1]
    if hasattr(model, "decision_function"):
        scores = model.decision_function(X_eval)
        return (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)
    return None


def classifier_params(model):
    classifier = model.named_steps["classifier"]
    params = {"classifier": classifier.__class__.__name__}
    if hasattr(classifier, "n_neighbors"):
        params["n_neighbors"] = classifier.n_neighbors
    if hasattr(classifier, "kernel"):
        params["kernel"] = classifier.kernel
    if hasattr(classifier, "C"):
        params["C"] = classifier.C
    return params


results = []
predictions = {}
for name, model in classical_models.items():
    model.fit(X_train_raw, y_train)
    y_pred = model.predict(X_test_raw)
    y_score = positive_scores(model, X_test_raw)
    auc = roc_auc_score(y_test, y_score) if y_score is not None and len(np.unique(y_test)) == 2 else np.nan
    row = {
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": auc,
    }
    row.update(classifier_params(model))
    results.append(row)
    predictions[name] = y_pred

classical_results = pd.DataFrame(results).sort_values("balanced_accuracy", ascending=False)
display(classical_results)

### 11.2 DNN sencilla

La DNN tambien usa los embeddings originales aplanados. Solo se aplica `StandardScaler` ajustado con train. Es intencionalmente pequena y regularizada para reducir sobreajuste, porque el numero de sujetos es bajo frente al numero de features.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

dnn_scaler = StandardScaler()
X_train_scaled = dnn_scaler.fit_transform(X_train_raw)
X_test_scaled = dnn_scaler.transform(X_test_raw)

X_train_dnn = X_train_scaled.astype(np.float32)
X_test_dnn = X_test_scaled.astype(np.float32)

class EmbeddingMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        hidden = 128
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.ReLU(),
            nn.Dropout(0.35),
            nn.Linear(hidden, 32),
            nn.ReLU(),
            nn.Dropout(0.35),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(1)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = EmbeddingMLP(X_train_dnn.shape[1]).to(device)

class_counts = np.bincount(y_train.astype(int), minlength=2)
pos_weight = torch.tensor([class_counts[0] / max(class_counts[1], 1)], dtype=torch.float32, device=device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)

train_ds = TensorDataset(
    torch.tensor(X_train_dnn, dtype=torch.float32),
    torch.tensor(y_train.astype(np.float32), dtype=torch.float32),
)
train_loader = DataLoader(train_ds, batch_size=min(16, len(train_ds)), shuffle=True)

best_state = None
best_loss = float("inf")
patience = 30
bad_epochs = 0
history = []

X_train_tensor = torch.tensor(X_train_dnn, dtype=torch.float32, device=device)
X_test_tensor = torch.tensor(X_test_dnn, dtype=torch.float32, device=device)
y_train_tensor = torch.tensor(y_train.astype(np.float32), dtype=torch.float32, device=device)
y_test_tensor = torch.tensor(y_test.astype(np.float32), dtype=torch.float32, device=device)

for epoch in range(300):
    model.train()
    epoch_losses = []
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        epoch_losses.append(loss.item())

    model.eval()
    with torch.no_grad():
        train_logits = model(X_train_tensor)
        test_logits = model(X_test_tensor)
        test_loss = criterion(test_logits, y_test_tensor).item()

        train_preds = (torch.sigmoid(train_logits) >= 0.5).cpu().numpy().astype(int)
        test_preds = (torch.sigmoid(test_logits) >= 0.5).cpu().numpy().astype(int)

    train_loss = float(np.mean(epoch_losses))
    train_accuracy = accuracy_score(y_train, train_preds)
    test_accuracy = accuracy_score(y_test, test_preds)

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "test_loss": test_loss,
        "train_accuracy": train_accuracy,
        "test_accuracy": test_accuracy,
    })

    if test_loss < best_loss:
        best_loss = test_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        bad_epochs = 0
    else:
        bad_epochs += 1
        if bad_epochs >= patience:
            break

if best_state is not None:
    model.load_state_dict(best_state)

history_df = pd.DataFrame(history)
display(history_df.tail())

plt.figure(figsize=(7, 4))
plt.plot(history_df["epoch"], history_df["train_loss"], label="train")
plt.plot(history_df["epoch"], history_df["test_loss"], label="test")
plt.xlabel("epoch")
plt.ylabel("BCE loss")
plt.title("Curva de entrenamiento DNN")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(history_df["epoch"], history_df["train_accuracy"], label="train_accuracy")
plt.plot(history_df["epoch"], history_df["test_accuracy"], label="test_accuracy")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.title("Curva de accuracy DNN")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
model.eval()
with torch.no_grad():
    logits = model(X_test_tensor)
    dnn_scores = torch.sigmoid(logits).detach().cpu().numpy()
dnn_pred = (dnn_scores >= 0.5).astype(int)

dnn_result = pd.DataFrame([{
    "model": "DNN_simple",
    "accuracy": accuracy_score(y_test, dnn_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, dnn_pred),
    "f1": f1_score(y_test, dnn_pred),
    "roc_auc": roc_auc_score(y_test, dnn_scores) if len(np.unique(y_test)) == 2 else np.nan,
}])

all_classification_results = pd.concat([classical_results, dnn_result], ignore_index=True)
display(all_classification_results.sort_values("balanced_accuracy", ascending=False))

print("Reporte DNN:")
print(classification_report(y_test, dnn_pred, target_names=[LABEL_NAMES[0], LABEL_NAMES[1]]))

cm = confusion_matrix(y_test, dnn_pred)
plt.figure(figsize=(4, 3))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=[LABEL_NAMES[0], LABEL_NAMES[1]], yticklabels=[LABEL_NAMES[0], LABEL_NAMES[1]])
plt.xlabel("Prediccion")
plt.ylabel("Real")
plt.title("Matriz de confusion DNN")
plt.tight_layout()
plt.show()

### 11.3 Guardar resultados de clasificacion

In [ ]:
classification_out = OUT_DIR / "classification"
classification_out.mkdir(parents=True, exist_ok=True)

all_classification_results.to_csv(classification_out / "classification_metrics.csv", index=False)
history_df.to_csv(classification_out / "dnn_training_history.csv", index=False)

test_predictions = meta_test[["ID", "label", "class_name"]].copy().reset_index(drop=True)
if "split" in meta_test.columns:
    test_predictions["original_split"] = meta_test["split"].values
for name, pred in predictions.items():
    test_predictions[f"pred_{name}"] = pred
test_predictions["pred_DNN_simple"] = dnn_pred
test_predictions["score_DNN_simple"] = dnn_scores
test_predictions.to_csv(classification_out / "test_predictions.csv", index=False)

print("Resultados de clasificacion guardados en:", classification_out)